# Download validation

Questo notebook controlla download log, link validati e registry. Le celle servono a scegliere quali URL sono candidati a download automatico e quali restano revisione manuale.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display

for path in [Path.cwd(), *Path.cwd().parents]:
    helper_path = path / "notebooks" / "utils_notebooks.py"
    if helper_path.exists():
        sys.path.insert(0, str(helper_path.parent))
        break

from utils_notebooks import get_project_paths, plot_barh, plot_matrix, read_clean_csv, show_table

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 140)

PATHS = get_project_paths()
ROOT = PATHS["root"]
CATALOG_DIR = PATHS["catalog"]
TABLES = PATHS["tables"]
METADATA = PATHS["metadata"]
PROCESSED = PATHS["processed"]
print("Project root:", ROOT)

## Caricamento tabelle

La pipeline puo' non scaricare file quando le fonti sono disabilitate o richiedono revisione manuale. In quel caso il notebook valuta comunque discovery e validazione link.

In [ ]:
log = read_clean_csv(METADATA / "downloads_log.csv")
discovered = read_clean_csv(CATALOG_DIR / "discovered_links.csv")
validated = read_clean_csv(TABLES / "validated_discovered_links.csv")
registry = read_clean_csv(TABLES / "dataset_registry.csv")

summary = pd.DataFrame([
    {"area": "download log", "rows": len(log)},
    {"area": "discovered links", "rows": len(discovered)},
    {"area": "validated links", "rows": len(validated)},
    {"area": "dataset registry", "rows": len(registry)},
])
display(summary)
show_table(log, 20)

## Esito download

Contiamo download riusciti e falliti. Quando non ci sono successi, leggere insieme a `download_status` e `enabled` nel catalogo.

In [ ]:
if not log.empty and "success" in log.columns:
    log["success_label"] = log["success"].astype(str)
    success_counts = log.groupby("success_label", dropna=False).size().reset_index(name="downloads")
    display(success_counts)
    plot_barh(success_counts, "success_label", "downloads", "Esito download", color="#0f766e")
    if "error_message" in log.columns:
        errors = log[log["error_message"].fillna("") != ""]
        show_table(errors[[column for column in ["download_url", "http_status", "error_message"] if column in errors.columns]], 20)
else:
    print("Nessun download diretto eseguito o colonna success non presente.")

## Validazione link

La validazione distingue URL raggiungibili, errori, formati dati e report.

In [ ]:
if not validated.empty:
    status_counts = validated.groupby("validation_status_code", dropna=False).size().reset_index(name="links")
    status_counts["validation_status_code"] = status_counts["validation_status_code"].astype(str)
    display(status_counts.sort_values("links", ascending=False))
    plot_barh(status_counts, "validation_status_code", "links", "Link per status code di validazione", color="#2563eb")

    data_series = validated.get("is_data_file", pd.Series(False, index=validated.index)).fillna(False).astype(bool)
    report_series = validated.get("is_report_file", pd.Series(False, index=validated.index)).fillna(False).astype(bool)
    type_rows = pd.DataFrame([
        {"type": "data_file", "links": int(data_series.sum())},
        {"type": "report_file", "links": int(report_series.sum())},
        {"type": "other", "links": int((~data_series & ~report_series).sum())},
    ])
    display(type_rows)
    plot_barh(type_rows, "type", "links", "Tipologia link validati", color="#7c3aed")
else:
    print("Validazione link non disponibile.")

In [ ]:
if not validated.empty:
    provider_validation = validated.groupby(["provider", "validation_status_code"], dropna=False).size().reset_index(name="links")
    provider_totals = provider_validation.groupby("provider", dropna=False)["links"].sum().reset_index()
    plot_barh(provider_totals, "provider", "links", "Link validati per provider", color="#0f766e")
    show_table(provider_validation.sort_values(["provider", "links"], ascending=[True, False]), 40)

    extension_counts = validated.groupby("file_extension", dropna=False).size().reset_index(name="links")
    plot_barh(extension_counts, "file_extension", "links", "Formati candidati", color="#b45309")

## Errori e candidati utili

Usare queste tabelle per separare URL instabili da candidati dati con status raggiungibile.

In [ ]:
if not validated.empty:
    error_rows = validated[validated.get("validation_error", pd.Series("", index=validated.index)).fillna("") != ""].copy()
    print("Link con errore di validazione:", len(error_rows))
    show_table(error_rows[[column for column in ["provider", "source_id", "found_url", "validation_error"] if column in error_rows.columns]], 20)

if not registry.empty:
    registry["is_data_file_bool"] = registry.get("is_data_file", pd.Series(False, index=registry.index)).fillna(False).astype(bool)
    data_candidates = registry[registry["is_data_file_bool"]].copy()
    if "validation_status_code" in data_candidates.columns:
        data_candidates["validation_status_code"] = pd.to_numeric(data_candidates["validation_status_code"], errors="coerce")
        data_candidates = data_candidates.sort_values(["validation_status_code", "provider", "source_id"], na_position="last")
    columns = ["provider", "source_id", "dataset_name", "theme", "file_extension", "validation_status_code", "license", "candidate_url"]
    show_table(data_candidates[[column for column in columns if column in data_candidates.columns]], 25)
else:
    print("Registry dataset non disponibile.")

## Coda download

Tabella filtrabile per decidere quali URL promuovere a download configurato.

In [ ]:
if not registry.empty:
    queue = registry.copy()
    queue["is_data_file_bool"] = queue.get("is_data_file", pd.Series(False, index=queue.index)).fillna(False).astype(bool)
    queue["status_numeric"] = pd.to_numeric(queue.get("validation_status_code", pd.Series(np.nan, index=queue.index)), errors="coerce")
    queue["download_priority"] = queue["is_data_file_bool"].astype(int) * 10 + queue["status_numeric"].eq(200).astype(int) * 5
    columns = ["download_priority", "provider", "source_id", "dataset_name", "theme", "file_extension", "status_numeric", "license", "redistribution_allowed", "candidate_url"]
    columns = [column for column in columns if column in queue.columns]
    display(queue[columns].sort_values(["download_priority", "provider", "source_id"], ascending=[False, True, True]).head(80))